In [1]:
import glob
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
import mlflow
import config

In [2]:
# define Mlflow env. settings
mlflow.set_tracking_uri(uri=config.mlflow_uri)
%env MLFLOW_TRACKING_URI= config.mlflow_db
%env mlflow_tracking_username = config.mlflow_un
%env mlflow_tracking_password = config.mlflow_pw
#mlflow.enable_system_metrics_logging()

env: MLFLOW_TRACKING_URI=config.mlflow_db
env: mlflow_tracking_username=config.mlflow_un
env: mlflow_tracking_password=config.mlflow_pw


In [3]:
customerList = ["X5","X7","X8"]#,"X3","X4","X1","X2","X6"] #
#Replace with dynamic code later Not full list of audit types
#FilterTypeList = ['AddTest','Aliquot','Authorize','CalculationUpdate','Cancel','CancelResult','CancelSample','CancelTest','CreateQCSample','EditTests','FileImport','ModifyBaseDate','ModifySample','Prepare','ReactivateSample','ReceiveSample','Reject','ReleaseSample','RemoveTest','ReSample','RestoreResult','RestoreSample','RestoreTest','ReviewAuditTrail','Reviewed','TableUpdate','UnReceiveSample','UpdateBatch','UpdateLot'] 
FilterTypeList = ['AddTest','Aliquot','CalculationUpdate','CancelResult','CancelSample','CancelTest','ReactivateSample','Reject','RemoveTest','ReSample','RestoreResult','RestoreSample','RestoreTest','ReviewAuditTrail','Reviewed','UpdateBatch','UpdateLot'] 
#FilterTypeList = ['Aliquot','CalculationUpdate']
#'Authorize','EditTests','EnterResults','LogSample','ModifySample','ReceiveSample' - MemoryError: Unable to allocate 
#'Cancel','CreateQCSample','FileImport','ModifyBaseDate','Prepare','ReleaseSample','TableUpdate','UnReceiveSample' - No objects to concatenate
#mlflow.autolog()

In [4]:
# Start an MLflow run
mlflow.set_experiment("audit_type_based_testing")
algoType = "Sklearn_random_forest_holdout"

with mlflow.start_run(run_name=algoType, nested=True) as grandparent_run:
    for FilterType in FilterTypeList: 
        with mlflow.start_run(run_name=f"Audit_Type_{FilterType}", nested=True) as parent_run:
            files = glob.glob(f'C:/MLAR_DATA/T0001/FS_audit_type_table_name/{FilterType}/FS_audit_type_table_name_{FilterType}_X*.parquet')
            for holdoutCustomer in customerList:
                with mlflow.start_run(run_name=f"Run_exclude{holdoutCustomer}", nested=True) as child_run:
                    if any(holdoutCustomer in substring for substring in files): 
                        filtered_files = [word for word in files if not word.endswith(holdoutCustomer+'.parquet')]
                        df = pd.concat([pd.read_parquet(fp) for fp in filtered_files])
                        df = df.drop(['T0083','TEXT_ID_JSON'],axis=1) #reduce some of customer specific ID-related features already removed 'LABEL_ID','LABEL_MASK'
                        df = df.astype(str).replace('nan',np.nan).dropna(how='all')
                        features = df.columns.tolist()
                        categoricalColumns = df.drop(['T0083_SEVERITY'],axis=1).select_dtypes(include=['object']).columns.tolist()
                        df = pd.get_dummies(df[features], columns=categoricalColumns)
                        X_train, X_test, y_train, y_test = train_test_split(df[df.columns.drop(list(df.filter(like='T0083_SEVERITY')))], df.filter(like='T0083_SEVERITY').sum(axis=1), test_size=0.30, random_state=123)
                        
                        # classWeight = "balanced"
                        maxDepth = 5
                        # mlflow.log_param("class_weight",classWeight)
                        mlflow.log_param("max_depth",maxDepth)
                        
                        model = RandomForestClassifier(max_depth=maxDepth)
                        model.fit(X_train.values,y_train.values)

                        predictions = pd.DataFrame(model.predict(X_test.values))
                    
                        # Print and Log metrics
                        precisionScore = metrics.precision_score(y_test, predictions, average='micro',zero_division=1)
                        recallScore = metrics.recall_score(y_test, predictions, average='micro',zero_division=1)
                        f1Score = metrics.f1_score(y_test, predictions, average='micro',zero_division=1)               
                        classificationReport = metrics.classification_report(y_test, predictions,zero_division=1)

                        mlflow.log_metric("precision", precisionScore)
                        mlflow.log_metric("recall", recallScore)
                        mlflow.log_metric("f1_score", f1Score)
                        mlflow.log_text(classificationReport, "classificationReport.txt")
                        #training_dataset = mlflow.data.from_pandas(df[df.columns.drop(list(df.filter(like='T0083_SEVERITY')))])
                        #tagDict = {"Dataset":"training","excluded customer":{holdoutCustomer}}
                        #mlflow.log_input(X_train, context="training",tags=tagDict)
                        #testing_dataset = mlflow.data.from_pandas(df.filter(like='T0083_SEVERITY').sum(axis=1))
                        #tagDict = {"Dataset":"testing","excluded customer":{holdoutCustomer}}
                        #mlflow.log_input(X_test,context="testing",tags=tagDict)  
                        mlflow.log_table(predictions,"predictions.json")
                        
                        #model_uri = f"runs:/{run.info.run_id}/{algoType}/{FilterType}"
                        #model_name = f"{algoType}_{FilterType}_exclude{holdoutCustomer}"
                        #tagDict = {"algorithm type":{algoType},"Filter Type":{FilterType},"excluded customer":{holdoutCustomer}}
                        #mlflow.register_model(model_uri, model_name, tags=tagDict)   
                    else:
                        print(f"{holdoutCustomer} not present for {FilterType}.")   


X8 not present for Aliquot.
X8 not present for CancelResult.
X5 not present for RestoreResult.
X5 not present for UpdateLot.
